### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [1]:
import os
# to use our OPENAI API key
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
# from langchain.schema import Document - older legacy

# vector store imports
# from langchain_community.vectorstores import Chroma - legacy
from langchain_chroma import Chroma

# utility imports
import numpy as np
from typing import List


##### WE are not creating sample data again , as already done in previous step , and files are available in data directory

###  Document Loading

In [3]:
from langchain_community.document_loaders import DirectoryLoader , TextLoader 

# Load the documents from directory
doc_loader = DirectoryLoader(
    path="data",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    glob="*.txt"

)

documents = doc_loader.load()

print(f"Loaded {len(documents)} documents")
print("Preview of First Doc:\n")
print(documents[0].page_content)

Loaded 3 documents
Preview of First Doc:


    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    


In [4]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natur

### Document Splitting

In [5]:
# Initializr Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    separators=[" "],
    length_function = len
)

chunks = text_splitter.split_documents(documents)

print(f" Created {len(chunks)} documents from {len(documents)} documents")
print(f"\nChunk Example:")
print(f"\nContent: {chunks[0].page_content}")
print(f"\nMeta Data: {chunks[0].metadata}")

 Created 5 documents from 3 documents

Chunk Example:

Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through

Meta Data: {'source': 'data\\doc_0.txt'}


In [6]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

### Embedding Models

In [7]:
os.environ['OPENAI_ENV_KEY'] = os.getenv('OPENAI_API_KEY')

In [8]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000164C294BA10>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000164C3E44440>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [9]:
# Create a cromadb vector store
persist_directory = "./chroma_db"

# Initialize chromadb with openAI embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=persist_directory,
    collection_name="rag_collection2"
)

print(f" Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

 Vector store created with 10 vectors
Persisted to : ./chroma_db


### Test Similrity Search

In [10]:
query = "What are the types of machine learning?"

similar_docs = vectorstore.similarity_search(query , k=3)
similar_docs

[Document(id='390fcf9e-697d-4e93-8d36-d4b61a7c2d51', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(id='6aa5d46e-c8e9-4346-b3ee-9163630d50a7', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and 

In [11]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What are the types of machine learning?

Top 3 similar chunks:

--- Chunk 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
Source: data\doc_0.txt

--- Chunk 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...
Source: data\doc_0.txt

--- Chunk 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt


In [12]:
# similarity search with score
result_score = vectorstore.similarity_search_with_score(query , k=3)
result_score

[(Document(id='390fcf9e-697d-4e93-8d36-d4b61a7c2d51', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  0.7797253131866455),
 (Document(id='6aa5d46e-c8e9-4346-b3ee-9163630d50a7', metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, un

Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18", temperature=0.0 , max_tokens=500)
# # use max_completed_tokens=500 if you want to limit the response length for newer models

In [ ]:
# from langchain.chat_models.base import init_chat_model
# llm = init_chat_model(model="gpt-4o-mini-2024-07-18", temperature=0.0 , max_tokens=500)

In [13]:
from langchain.chat_models.base import init_chat_model

# Explicitly defining the model provider
llm = init_chat_model(model="gpt-4o-mini-2024-07-18", model_provider="openai", temperature=0.0, max_tokens=500)


In [14]:
llm.invoke("What is AI")

AIMessage(content='Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. These processes include learning (the acquisition of information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI can be categorized into several types:\n\n1. **Narrow AI**: Also known as weak AI, this type is designed and trained for a specific task, such as language translation, facial recognition, or playing chess. Most AI applications today fall into this category.\n\n2. **General AI**: Also known as strong AI or AGI (Artificial General Intelligence), this type would have the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human being. As of now, AGI remains largely theoretical and has not yet been achieved.\n\n3. **Superintelligent AI**: This refers to a hypothetical AI that surpasses human intelligence across virtuall

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [16]:
# Create a custom prompt
from langchain_core.prompts import ChatPromptTemplate
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [17]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwargs={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000164C3E44590>, search_kwargs={'k': 3})

In [18]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [19]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000164C3E44590>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x00000164C3E47A10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000

In [20]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep learning has significantly impacted various fields, including computer vision, natural language processing, and speech recognition. Specific types of neural networks used in deep learning include Convolutional Neural Networks (CNNs), which are particularly effective for image processing, and Recurrent Neural Networks (RNNs) and Transformers, which are used for sequential data processing.'

In [34]:
# retriever.get_relevant_documents("What is Deep Learning") 
# depeceated

In [21]:
# The modern way to fetch relevant chunks manually
retriever.invoke("What is Deep Learning")

[Document(id='e540a8d7-e0bb-479d-ac4d-d1a32adfdd66', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(id='051faebf-607a-4fd7-a0c1-5ab85f327d97', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer 

In [22]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    # docs = retriever.get_relevant_documents(question) -- deprecated
    docs = retriever.invoke(question)

    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [23]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: The key concepts in reinforcement learning include:

1. **Interaction with an Environment**: Reinforcement learning involves an agent that interacts with an environment to learn how to achieve a goal.

2. **Rewards and Penalties**: The learning process is driven by feedback in the form of rewards (positive reinforcement) and penalties (negative reinforcement) based on the agent's actions.

These concepts highlight how reinforcement learning enables an agent to learn optimal behaviors through trial and error, guided by the outcomes of its actions.

Source Documents:

--- Source 1 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 3 ---
Machin

In [24]:
query_rag_lcel("What is machine learning?")

Question: What is machine learning?
--------------------------------------------------
Answer: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, which uses labeled data to train models; unsupervised learning, which finds patterns in unlabeled data; and reinforcement learning, which learns through interactions with an environment.

Source Documents:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are...

--- Source 3 ---
Deep Learning and Neural Networks



In [26]:
query_rag_lcel("What is CNN?")

Question: What is CNN?
--------------------------------------------------
Answer: CNN stands for Convolutional Neural Network. It is a type of deep learning architecture that is particularly effective for image processing. CNNs are designed to automatically and adaptively learn spatial hierarchies of features from images, making them well-suited for tasks in computer vision.

Source Documents:

--- Source 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...

--- Source 3 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key ta

In [25]:
query_rag_lcel("What is gradient descent?")

Question: What is gradient descent?
--------------------------------------------------
Answer: I don't know. The provided context does not include any information about gradient descent.

Source Documents:

--- Source 1 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 2 ---
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties....

--- Source 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...


### Add New Documents To Existing Vector Store

In [27]:
vectorstore

In [28]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [29]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n'

In [31]:
new_doc = Document(
    page_content=new_document,
    metadata = {"source":"manual_addition" , "topic" : "RL"}
)
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'RL'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [32]:
# split the new documents 
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'RL'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'RL'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [33]:
# adding to vector store
vectorstore.add_documents(new_chunks)

['ceed590b-7d37-4969-a106-25c88e3ab381',
 '743e1bdc-2b4e-4505-912d-5ae91d221143']

In [34]:
print(f"Added {len(new_chunks) } new chunks to the vector store")
print(f"Present total vectors: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Present total vectors: 12


In [36]:
# query with the pdated vector store
new_query = "What are the key compnents of RL?"
results = query_rag_lcel(new_query)
results

Question: What are the key compnents of RL?
--------------------------------------------------
Answer: The key components of reinforcement learning (RL) include:

1. **States**: These represent the different situations or configurations that the agent can encounter in the environment.
2. **Actions**: These are the choices or moves that the agent can make in response to the current state.
3. **Rewards**: These are the feedback signals received by the agent after taking an action, which can be positive (reward) or negative (penalty).
4. **Policies**: These are the strategies or rules that the agent follows to determine which action to take in a given state.
5. **Value Functions**: These estimate the expected cumulative reward that can be obtained from a given state or state-action pair.

These components work together to enable the agent to learn and make decisions that maximize cumulative rewards over time.

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforce

In [37]:
# query with the pdated vector store
new_query = "Difference between RL , ML , DL and NLP. Provide respnse in List or tabular format"
results = query_rag_lcel(new_query)
results

Question: Difference between RL , ML , DL and NLP. Provide respnse in List or tabular format
--------------------------------------------------
Answer: Here is a comparison of Reinforcement Learning (RL), Machine Learning (ML), Deep Learning (DL), and Natural Language Processing (NLP) in a tabular format:

| Aspect                     | Reinforcement Learning (RL)                                      | Machine Learning (ML)                                      | Deep Learning (DL)                                         | Natural Language Processing (NLP)                          |
|---------------------------|------------------------------------------------------------------|-----------------------------------------------------------|-----------------------------------------------------------|-----------------------------------------------------------|
| Definition                 | A type of ML where an agent learns by interacting with an environment to maximize cumulative rewards. |

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:
<ul>
<li>Follow-up questions that reference previous answersM</li>
<li>Pronoun resolution (e.g., "it", "they", "that")</li>
<li>Context-dependent queries that build on prior discussion</li>
<li>Natural dialogue flow where users don't repeat context</li>
</ul>

**Key Challenge :**
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"<br>
Bot: explains Python programming language<br>
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this<br>

Solution:
The modern approach uses a two-step process:

- **Query Reformulation :** Transform context-dependent questions into standalone queries
- **Context-Aware Retrieval :** Use the reformulated query to fetch relevant documents


- **create_history_aware_retriever :** Makes the retriever understand conversation context --- **Depreceated**
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [39]:
# from langchain.chains import create_history_aware_retriever
# # from langchain.chains import create_history_aware_retriever - deprecated 
# from langchain_core.prompts import MessagesPlaceholder
# from langchain_core.messages import HumanMessage, AIMessage

ModuleNotFoundError: No module named 'langchain.chains'

Building a dynamic, native LCEL pipeline that explicitly handles query reformulation based on conversation history.A more clean approach then previous drecepated approach

In [63]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [64]:
# 1. Define the system prompt instruction for rewriting
contextualize_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

# 2. Build the history-aware prompt template
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# 3. Define the sub-chain that handles rewriting (runs if history exists)
query_rewriter = contextualize_prompt | llm | StrOutputParser()

# 4. Create a dynamic path: if chat history exists, pass the rewritten query to the retriever. 
# Otherwise, pass the raw input string directly to the retriever.
def route_retrieval(input_dict):
    if input_dict.get("chat_history"):
        return query_rewriter | retriever
    return lambda x: retriever.invoke(x["input"])

history_aware_retriever = RunnablePassthrough() | route_retrieval

In [42]:
# 5. Define the QA generation prompt
qa_system_prompt = (
    "Use the following context to answer the question. \n"
    "If you don't know the answer based on the context, say you don't know.\n"
    "Provide specific details from the context to support your answer.\n\n"
    "Context:\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# 6. Assemble the full runnable chain
conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=history_aware_retriever | format_docs
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

print("Conversational LCEL Chain initialized successfully!")

Conversational LCEL Chain initialized successfully!


In [65]:
# Simulate a conversational thread
chat_history = [
    HumanMessage(content="What is Machine Learning?"),
    AIMessage(content="Machine learning is a subset of AI that allows systems to learn from experience without explicit programming.")
]

# Follow up with a relative pronoun query
new_question = "What are its three main types?"

print(f"User Question: {new_question}\n")

response = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": new_question
})

print(f"Answer:\n{response}")

User Question: What are its three main types?

Answer:
The three main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interactions with the environment.


### Using GROQ LLM's

In [51]:
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x00000164C3E47A10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000164C3E47620>, root_client=<openai.OpenAI object at 0x00000164C3E4F9D0>, root_async_client=<openai.AsyncOpenAI object at 0x00000164C4528050>, model_name='gpt-4o-mini-2024-07-18', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True, max_tokens=500)

In [52]:
load_dotenv()

True

In [56]:
# os.getenv("GROQ_API_KEY")

In [54]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [55]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [62]:
llm=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct",api_key=os.getenv("GROQ_API_KEY"))
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000164C73B6E90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000164C73B7390>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)